In [1]:
import sys
sys.path.insert(0, "/projappl/project_2012747/mars/MarS")  # folder that contains market_simulation/
from market_simulation.models.order_model import OrderModel


2026-01-09 15:38:05,024 - /projappl/project_2012747/mars/MarS/market_simulation/__init__.py:15 - INFO - init logging


/PUHTI_TYKKY_Quvj2Tb/miniforge/envs/env1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import IterableDataset, DataLoader

from datasets import load_dataset
from market_simulation.models.order_model import OrderModel


# -------------------------
# Config
# -------------------------
K = 1024
TRAIN_STRIDE = 16
VAL_STRIDE = 16
VAL_FRACTION = 0.10

BATCH_SIZE = 8
LR = 3e-4
MAX_STEPS = 20_000
EVAL_EVERY = 50
VAL_MAX_BATCHES = 200  # keep frequent eval cheap; set None for full val

EMB_DIM = 64
NUM_LAYERS = 2
NUM_HEADS = 4

USE_AMP = True
AMP_DTYPE = torch.bfloat16  # or torch.float16

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feat_cols = [f"f{i}" for i in range(15)]


# -------------------------
# Load with 🤗 Datasets
# -------------------------
hf = load_dataset("parquet", data_files={"data": "../data/mymessages.parquet"})["data"]

def add_f4(batch):
    """ convert ns since midnight into s since market open (9h30)
        and clip it to maximum 6h30 hours (time of the open market, ie from 9h30 to 16h)
    """
    t = np.asarray(batch["Time"], dtype=np.int64)
    t_sec = (t // 1_000_000_000).astype(np.int64)
    batch["f4"] = np.clip(t_sec - 34200, 0, 23399).astype(np.int64)
    return batch

# we use .map function from Huggingface, to compute the f4 column faster (i.e. we compute it by "batches")
hf = hf.map(add_f4, batched=True, batch_size=200_000, num_proc=1)

# keep only f0..f14
drop_cols = [c for c in hf.column_names if c not in feat_cols]
if drop_cols:
    hf = hf.remove_columns(drop_cols)

T = len(hf)
split = int(T * (1.0 - VAL_FRACTION))
train_hf = hf.select(range(0, split))
val_hf   = hf.select(range(split, T))

print(f"Rows: {T:,} | train: {len(train_hf):,} | val: {len(val_hf):,}")

# Fast path: format columns as numpy arrays once
train_hf = train_hf.with_format("numpy")
val_hf   = val_hf.with_format("numpy")

train_cols = {c: train_hf[c] for c in feat_cols}  # each is a numpy array
val_cols   = {c: val_hf[c]   for c in feat_cols}



# -------------------------
# Iterable datasets that yield *batched* windows
# -------------------------
class BatchedWindowIterable(IterableDataset):
    """
    Yields pre-batched windows of shape (B, K, 15).

    If random=False:
        deterministic, strided windows (good for validation)
    If random=True:
        random windows sampled from a stride grid (good for training)
    """
    def __init__(
        self,
        cols: dict,
        K: int,
        stride: int,
        batch_size: int,
        random: bool = False,
        seed: int = 0,
    ):
        self.cols = cols
        self.K = K
        self.stride = stride
        self.batch_size = batch_size
        self.random = random
        self.seed = seed

        self.N = len(next(iter(cols.values())))
        self.max_start = self.N - K

        # allowed start positions
        self.grid = np.arange(0, self.max_start + 1, stride, dtype=np.int64)

    def __iter__(self):
        B = self.batch_size
        K = self.K
        cols = self.cols
        grid = self.grid
        feat_cols_local = feat_cols

        if self.random:
            # different seed per worker
            info = torch.utils.data.get_worker_info()
            worker_id = info.id if info else 0
            rng = np.random.default_rng(self.seed + worker_id)

            while True:
                s_batch = rng.choice(
                    grid,
                    size=B,
                    replace=(len(grid) < B),
                )

                X = np.empty((B, K, 15), dtype=np.int64)
                for bi, s in enumerate(s_batch):
                    for fj, c in enumerate(feat_cols_local):
                        X[bi, :, fj] = cols[c][s:s+K]

                yield torch.from_numpy(X).long()

        else:
            # deterministic, strided
            for i in range(0, len(grid), B):
                s_batch = grid[i:i+B]
                if len(s_batch) == 0:
                    break

                X = np.empty((len(s_batch), K, 15), dtype=np.int64)
                for bi, s in enumerate(s_batch):
                    for fj, c in enumerate(feat_cols_local):
                        X[bi, :, fj] = cols[c][s:s+K]

                yield torch.from_numpy(X).long()



train_iterable = BatchedWindowIterable(
    train_cols,
    K=K,
    stride=TRAIN_STRIDE,
    batch_size=BATCH_SIZE,
    random=True,
    seed=123,
)

train_dl = DataLoader(
    train_iterable,
    batch_size=None,
    num_workers=2,
    pin_memory=True,
)


val_iterable = BatchedWindowIterable(
    val_cols,
    K=K,
    stride=VAL_STRIDE,
    batch_size=BATCH_SIZE,
    random=False,
)

val_dl = DataLoader(
    val_iterable,
    batch_size=None,
    num_workers=2,
    pin_memory=True,
)



# -------------------------
# Model
# -------------------------
model = OrderModel(
    emb_dim=EMB_DIM,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    num_max_orders=K,
).to(device)

print(f"Model params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
opt = torch.optim.AdamW(model.parameters(), lr=LR)

use_scaler = (USE_AMP and device.type == "cuda" and AMP_DTYPE == torch.float16)
scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)


def lm_loss_all_positions(logits: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
    targets = X[:, :, 0]           # (B, K)
    logits_s = logits[:, :-1, :]   # (B, K-1, vocab)
    targ_s   = targets[:, 1:]      # (B, K-1)
    return F.cross_entropy(
        logits_s.reshape(-1, logits_s.size(-1)),
        targ_s.reshape(-1),
        reduction="mean",
    )


@torch.no_grad()
def compute_val_loss() -> float:
    model.eval()
    total = 0.0
    count = 0
    for b, X in enumerate(val_dl, start=1):
        X = X.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(USE_AMP and device.type == "cuda"), dtype=AMP_DTYPE):
            logits = model(X)   # expects full logits (no logits_to_keep)
            loss = lm_loss_all_positions(logits, X)

        n = X.size(0) * (X.size(1) - 1)
        total += loss.item() * n
        count += n

        if VAL_MAX_BATCHES is not None and b >= VAL_MAX_BATCHES:
            break

    return total / max(1, count)


# -------------------------
# Train: step-based + eval every 50 steps
# -------------------------
model.train()
train_it = iter(train_dl)

pbar = tqdm(range(1, MAX_STEPS + 1), desc="train steps")
for step in pbar:
    X = next(train_it)  # already a batch: (B, K, 15)
    X = X.to(device, non_blocking=True)

    opt.zero_grad(set_to_none=True)

    with torch.amp.autocast("cuda", enabled=(USE_AMP and device.type == "cuda"), dtype=AMP_DTYPE):
        logits = model(X)
        loss = lm_loss_all_positions(logits, X)

    if use_scaler:
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
    else:
        loss.backward()
        opt.step()

    pbar.set_postfix(train_loss=f"{loss.item():.4f}")

    if step % EVAL_EVERY == 0:
        val_loss = compute_val_loss()
        print(f"\nstep {step:6d} | val_loss {val_loss:.4f}\n")
        model.train()


Rows: 2,312,305 | train: 2,081,074 | val: 231,231
Model params: 10.18M


train steps:   0%|          | 52/20000 [00:15<5:46:48,  1.04s/it, train_loss=9.5094]


step     50 | val_loss 9.5680



train steps:   1%|          | 102/20000 [00:29<5:45:46,  1.04s/it, train_loss=7.7069]


step    100 | val_loss 7.8735



...

train steps:  17%|█▋        | 3452/20000 [18:28<4:50:24,  1.05s/it, train_loss=3.0961]


step   3450 | val_loss 3.0197



train steps:  18%|█▊        | 3502/20000 [18:43<4:47:03,  1.04s/it, train_loss=2.6106]


step   3500 | val_loss 3.0143



train steps:  18%|█▊        | 3549/20000 [18:49<36:41,  7.47it/s, train_loss=2.0981]  